# NYC 311 | Thời gian xử lý yêu cầu ở Queens

**Chế độ chạy:** Notebook chọn CSV chính thức nếu có, nếu thiếu sẽ dùng bản mô phỏng. Kết quả lưu sẵn là kết quả mô phỏng.

**Câu hỏi:** Loại yêu cầu nào mất nhiều thời gian xử lý, và loại nào còn nhiều hồ sơ chưa đóng? Phạm vi là các yêu cầu tạo trong tháng 1/2025 tại Queens. Mốc chốt là đầu ngày 01/03/2025 theo giờ địa phương. [Nguồn NYC Open Data](https://data.cityofnewyork.us/Social-Services/311-Service-Requests-from-2020-to-Present/erm2-nwe9/data_preview)

## 1. Đọc và kiểm tra dữ liệu

Mỗi dòng là một yêu cầu. Giữ mã yêu cầu dưới dạng chuỗi và kiểm tra trùng mã, khoảng ngày tạo và ngày đóng bất hợp lệ.

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display, HTML

root = Path.cwd()
source = root / "data/queens-2025-01.csv"
demo = not source.exists()
if demo:
    source = root / "data/demo.csv"
print("DỮ LIỆU MÔ PHỎNG" if demo else "DỮ LIỆU NYC OPEN DATA")
data = pd.read_csv(source, dtype={"unique_key": "string"})
for column in ("created_date", "closed_date"):
    parsed = pd.to_datetime(data[column], format="ISO8601", errors="coerce")
    assert not (data[column].notna() & parsed.isna()).any(), column
    data[column] = parsed
assert len(data) and len(data) < 100_000
assert data.unique_key.notna().all() and data.unique_key.is_unique
assert data.borough.eq("QUEENS").all()
assert data.created_date.between("2025-01-01", "2025-02-01", inclusive="left").all()
bad = data.closed_date.lt(data.created_date)
print(f"{len(data):,} yêu cầu; {bad.sum():,} ngày đóng trước ngày tạo")
valid = data.loc[~bad].copy()
valid["complaint_type"] = valid.complaint_type.fillna("Không rõ")


DỮ LIỆU MÔ PHỎNG
600 yêu cầu; 0 ngày đóng trước ngày tạo


## 2. Tách hồ sơ theo ngày chốt

Hồ sơ có ngày đóng trước 01/03 được tính là đã đóng. Hồ sơ thiếu ngày đóng hoặc đóng từ 01/03 trở đi được tính là còn mở tại ngày chốt. Thời gian xử lý và tuổi hồ sơ đo theo ngày, tính từ thời điểm tạo.

In [2]:
cutoff = pd.Timestamp("2025-03-01")
closed_mask = valid.closed_date.lt(cutoff)
closed = valid.loc[closed_mask].copy()
opened = valid.loc[~closed_mask].copy()
closed["processing_days"] = (closed.closed_date - closed.created_date) / pd.Timedelta(days=1)
opened["age_days"] = (cutoff - opened.created_date) / pd.Timedelta(days=1)
assert len(closed) + len(opened) + bad.sum() == len(data)
opened["age_band"] = pd.cut(opened.age_days, [0, 7, 31, float("inf")],
                            labels=["<7 ngày", "7–30 ngày", "≥31 ngày"], right=False)
print(f"Đã đóng: {len(closed):,} | Còn mở: {len(opened):,}")
display(opened.age_band.value_counts(sort=False))


Đã đóng: 486 | Còn mở: 114


age_band
<7 ngày 0
7–30 ngày 20
≥31 ngày 94
Name: count, dtype: int64

## 3. So sánh theo loại yêu cầu

Chỉ xếp hạng thời gian xử lý cho nhóm có ít nhất 30 hồ sơ đã đóng. Trung vị là thời gian mà một nửa số hồ sơ đã đóng xử lý nhanh hơn hoặc bằng.

In [3]:
by_type = closed.groupby("complaint_type").processing_days.agg(
    closed_count="size", median_days="median")
slowest = by_type.query("closed_count >= 30").nlargest(10, "median_days")
backlog = opened.groupby(["complaint_type", "age_band"], observed=False).size().unstack(fill_value=0)
backlog["total"] = backlog.sum(axis=1)
largest = backlog.nlargest(10, "total")
display(slowest.round(2))
display(largest)
from html import escape

def bars(series, title):
    if series.empty:
        print("Không đủ hồ sơ để vẽ:", title)
        return
    maximum = max(float(series.max()), 1)
    rows = []
    for i, (label, value) in enumerate(series.items()):
        width = 320 * value / maximum
        y = i * 28 + 45
        rows.append(f'<text x="0" y="{y+15}">{escape(str(label)[:42])}</text>'
                    f'<rect x="320" y="{y}" width="{width}" height="19" fill="#167d9a"/>'
                    f'<text x="{328+width}" y="{y+15}">{value:.2f}</text>')
    caption = escape(title + (" · MÔ PHỎNG" if demo else ""))
    display(HTML(f'<svg role="img" aria-label="{caption}" width="740" height="{len(rows)*28+55}" '
                 f'font-family="sans-serif" font-size="13"><text x="0" y="22">{caption}</text>'
                 + ''.join(rows) + '</svg>'))

bars(slowest.median_days, "Trung vị thời gian xử lý (ngày)")
bars(largest.total, "Hồ sơ còn mở (số yêu cầu)")


,closed_count,median_days
complaint_type,,
Plumbing,114,20.38
Heat/Hot Water,94,17.48
Street Condition,141,10.46
Noise,137,1.46


age_band,<7 ngày,7–30 ngày,≥31 ngày,total
complaint_type,,,,
Heat/Hot Water,0,6,50,56
Plumbing,0,10,26,36
Noise,0,2,11,13
Street Condition,0,2,7,9


Trung vị thời gian xử lý (ngày) · MÔ PHỎNG Plumbing 20.38 Heat/Hot Water 17.48 Street Condition 10.46 Noise 1.46

Hồ sơ còn mở (số yêu cầu) · MÔ PHỎNG Heat/Hot Water 56.00 Plumbing 36.00 Noise 13.00 Street Condition 9.00

## 4. Cách diễn giải

Đọc hai bảng cùng nhau: loại có trung vị cao nhất và loại còn nhiều hồ sơ nhất có thể khác nhau. Tất cả hồ sơ trong nhóm tháng 1 đã có 28–59 ngày theo dõi ở mốc chốt, nên nhóm dưới 7 ngày sẽ bằng 0. Tồn đọng ở đây chỉ xét yêu cầu được tạo trong tháng 1 tại Queens. Trạng thái được suy từ ngày đóng trong bản xuất hiện tại, nên không thể tái dựng lịch sử mở lại hồ sơ. Trung vị chỉ mô tả nhóm đã đóng và có thể thấp nếu các hồ sơ khó vẫn còn mở.